<a href="https://colab.research.google.com/github/Shrideshi1/multi-label-email-risk-detection/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Confidential_Risk_Project"
DATA_RAW_DIR = f"{PROJECT_DIR}/data/raw"
DATA_PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
MODELS_DIR = f"{PROJECT_DIR}/models"
REPORTS_DIR = f"{PROJECT_DIR}/reports"

os.chdir(PROJECT_DIR)

print("Current folder:", os.getcwd())
print("Processed files:", os.listdir(DATA_PROCESSED_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current folder: /content/drive/MyDrive/Confidential_Risk_Project
Processed files: ['synthetic_confidential.csv', 'combined_dataset.csv', 'features']


In [ ]:
combined_df = pd.read_csv(f"{DATA_PROCESSED_DIR}/combined_dataset.csv")

print("Dataset shape:", combined_df.shape)
display(combined_df.head())

Dataset shape: (69910, 12)


,text,category,source,financial_risk,credential_risk,customer_info_risk,proprietary_risk,legal_risk,attachment_risk,phishing_spam_risk,project_category,risk_score
0,job posting - apple-iss research center conten...,ham,messages,0,0,0,0,0,0,0,Ham,0
1,"lang classification grimes , joseph e . and b...",ham,messages,0,0,0,0,0,0,0,Ham,0
2,query : letter frequencies for text identifica...,ham,messages,0,0,0,0,0,0,0,Ham,0
3,risk a colleague and i are researching the dif...,ham,messages,0,0,0,0,0,0,0,Ham,0
4,request book information earlier this morning ...,ham,messages,0,0,0,0,0,0,0,Ham,0


In [ ]:
risk_columns = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]

X_text = combined_df["text"].fillna("")
y = combined_df[risk_columns]

print("Text samples:", X_text.shape)
print("Label matrix:", y.shape)
display(y.sum().sort_values(ascending=False).rename("Label Count"))

Text samples: (69910,)
Label matrix: (69910, 7)


,Label Count
phishing_spam_risk,23073
legal_risk,21349
financial_risk,4014
customer_info_risk,1750
credential_risk,1750
attachment_risk,1750
proprietary_risk,1000


In [ ]:
print(combined_df.shape)
print(combined_df.columns.tolist())

(69910, 12)
['text', 'category', 'source', 'financial_risk', 'credential_risk', 'customer_info_risk', 'proprietary_risk', 'legal_risk', 'attachment_risk', 'phishing_spam_risk', 'project_category', 'risk_score']


In [ ]:
print("Missing values:")
print(combined_df[["text"] + risk_columns].isnull().sum())

print("Risk labels per row:")
display(
    y.sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("Number of Risk Labels")
    .reset_index(name="Rows")
)

Missing values:
text                  0
financial_risk        0
credential_risk       0
customer_info_risk    0
proprietary_risk      0
legal_risk            0
attachment_risk       0
phishing_spam_risk    0
dtype: int64
Risk labels per row:


,Number of Risk Labels,Rows
0,0,19724
1,1,45686
2,2,4500


In [ ]:
print("Dataset loaded successfully.")
print("X_text:", X_text.shape)
print("y:", y.shape)

Dataset loaded successfully.
X_text: (69910,)
y: (69910, 7)


In [ ]:
# ------------------------------------------------------------
# Text Cleaning
# ------------------------------------------------------------
# This function standardizes text before feature extraction.
# It lowercases all text, replaces URLs/emails/numbers with
# placeholders, removes unusual symbols, removes email formatting artifacts, long underscore
# separators, code-like underscore tokens, URLs, emails, and numbers.
# normalizes spacing.

import re

def clean_text(text):
    text = str(text).lower()

    #Replace URLs and email addresses with placeholders
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)

    #Remove long separator lines made of underscores/dashes/equal signs
    text = re.sub(r"[_\-=\*]{3,}", " ", text)

    #Replace code-style underscore tokens with spaces
    text = re.sub(r"\b_+\w+_*\b", " ", text)
    text = re.sub(r"\b\w+_+\w+\b", " ", text)

    #Replace numbers with a placeholder
    text = re.sub(r"\d+", " NUMBER ", text)

    #Keep only letters, numbers, spaces, periods, dollar signs, and dashes
    text = re.sub(r"[^a-zA-Z0-9\s\.\-$]", " ", text)

    #Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

combined_df["clean_text"] = combined_df["text"].apply(clean_text)

display(combined_df[["text", "clean_text"]].head())

,text,clean_text
0,job posting - apple-iss research center conten...,job posting - apple-iss research center conten...
1,"lang classification grimes , joseph e . and b...",lang classification grimes joseph e . and barb...
2,query : letter frequencies for text identifica...,query letter frequencies for text identificati...
3,risk a colleague and i are researching the dif...,risk a colleague and i are researching the dif...
4,request book information earlier this morning ...,request book information earlier this morning ...


In [ ]:
# ------------------------------------------------------------
# Metadata Features
# ------------------------------------------------------------
# Metadata features describe basic structural properties of each text.
# These features may help the model identify suspicious or sensitive
# messages based on length, numbers, symbols, and punctuation.

metadata_df = pd.DataFrame({
    #Total number of characters in the message
    "text_length": combined_df["text"].astype(str).str.len(),

    #Total number of words in the message
    "word_count": combined_df["text"].astype(str).str.split().str.len(),

    #Count of numeric digits, useful for IDs, account numbers, dates, and tokens
    "num_digits": combined_df["text"].astype(str).str.count(r"\d"),

    #Count of dollar signs, useful for financial content
    "num_dollar_signs": combined_df["text"].astype(str).str.count(r"\$"),

    #Count of uppercase letters, useful for emphasis or file-like text
    "num_uppercase": combined_df["text"].astype(str).str.count(r"[A-Z]"),

    #Count of exclamation marks, sometimes associated with urgency
    "num_exclamation": combined_df["text"].astype(str).str.count(r"!"),

    #Count of question marks
    "num_question": combined_df["text"].astype(str).str.count(r"\?")
})

# Preview metadata features
display(metadata_df.head())

,text_length,word_count,num_digits,num_dollar_signs,num_uppercase,num_exclamation,num_question
0,2896,590,32,1,0,0,0
1,1801,344,51,3,0,0,0
2,1486,287,60,0,0,1,0
3,329,61,0,0,0,0,1
4,1071,235,0,0,0,1,2


In [ ]:
# ------------------------------------------------------------
# Rule-Based Indicator Features
# ------------------------------------------------------------
# These binary features check whether each message contains terms
# associated with specific risk types. They are not the final labels,
# they are additional input features that may help the ML model.

text_lower = combined_df["text"].astype(str).str.lower()

rule_features_df = pd.DataFrame({
    #Detects common file extensions that may indicate attachments
    "has_attachment_ext": text_lower.str.contains(
        r"\.pdf|\.docx|\.xlsx|\.csv|\.zip|\.pptx",
        regex=True
    ).astype(int),

    #Detects financial language and currency symbols
    "has_money": text_lower.str.contains(
        r"\$| revenue | budget | valuation | invoice | payment ",
        regex=True
    ).astype(int),

    #Detects credential or access-related terms
    "has_credential_terms": text_lower.str.contains(
        r"password|token|api key|secret|credential|oauth|vpn|encryption key|root",
        regex=True
    ).astype(int),

    #Detects customer, client, employee, vendor, or account-related terms
    "has_customer_terms": text_lower.str.contains(
        r"customer|client|account|employee|vendor|partner|payroll",
        regex=True
    ).astype(int),

    #Detects legal or contract-related language
    "has_legal_terms": text_lower.str.contains(
        r"contract|nda|liability|clause|agreement|compliance|legal|indemnification",
        regex=True
    ).astype(int),

    #Detects internal/confidential routing language
    "has_internal_terms": text_lower.str.contains(
        r"internal|confidential|do not forward|do not distribute|not for circulation",
        regex=True
    ).astype(int)
})

# Preview rule-based features
display(rule_features_df.head())

# Show how often each rule-based feature appears
display(rule_features_df.sum().sort_values(ascending=False))

,has_attachment_ext,has_money,has_credential_terms,has_customer_terms,has_legal_terms,has_internal_terms
0,0,1,1,0,1,0
1,0,1,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


,0
has_legal_terms,10425
has_customer_terms,10086
has_money,6716
has_credential_terms,3914
has_attachment_ext,2082
has_internal_terms,1898


In [ ]:
# ------------------------------------------------------------
# TF-IDF Text Features
# ------------------------------------------------------------
# TF-IDF converts cleaned text into numerical features based on
# word importance. N-grams allow the model to learn single words
# and short phrases such as "api key", "do not", and "project atlas".

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    stop_words="english"
)

X_tfidf = tfidf_vectorizer.fit_transform(combined_df["clean_text"])

print("TF-IDF feature matrix shape:", X_tfidf.shape)
print("Number of TF-IDF features:", len(tfidf_vectorizer.get_feature_names_out()))

TF-IDF feature matrix shape: (69910, 10000)
Number of TF-IDF features: 10000


In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Sample TF-IDF features:")
print(feature_names[:50])

Sample TF-IDF features:
['aa' 'aa number' 'aaai' 'aaas' 'aamas' 'aaron' 'aaron kulkis' 'ab'
 'ab number' 'abc' 'ability' 'able' 'able use' 'abrahams' 'abs' 'absence'
 'absolute' 'absolutely' 'absolutely guaranteed' 'abstract'
 'abstract number' 'abstract submission' 'abstracts' 'abstracts number'
 'absurd' 'absurd episode' 'abuse' 'ac' 'ac number' 'ac uk' 'academia'
 'academic' 'academy' 'academy sciences' 'accent' 'accept' 'acceptable'
 'acceptance' 'acceptance notification' 'acceptance number' 'accepted'
 'accepted papers' 'accepting' 'accepts' 'access' 'access expires'
 'access token' 'accessed' 'accessible' 'accessing']


In [ ]:
# ------------------------------------------------------------
# Combine TF-IDF, Metadata, and Rule-Based Features
# ------------------------------------------------------------
# TF-IDF is sparse. Metadata and rule-based features are dense,
# so convert them to sparse matrices before combining.

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
import joblib
import os

#Scale metadata features so large numeric values like text_length

scaler = StandardScaler()

metadata_scaled = scaler.fit_transform(metadata_df)
metadata_sparse = csr_matrix(metadata_scaled)

rule_sparse = csr_matrix(rule_features_df.values)

#Final combined feature matrix
X_features = hstack([
    X_tfidf,
    metadata_sparse,
    rule_sparse
])

print("TF-IDF shape:", X_tfidf.shape)
print("Metadata shape:", metadata_sparse.shape)
print("Rule feature shape:", rule_sparse.shape)
print("Combined feature matrix shape:", X_features.shape)
print("Label matrix shape:", y.shape)

TF-IDF shape: (69910, 10000)
Metadata shape: (69910, 7)
Rule feature shape: (69910, 6)
Combined feature matrix shape: (69910, 10013)
Label matrix shape: (69910, 7)


In [ ]:
# ------------------------------------------------------------
# Save Feature Engineering Outputs
# ------------------------------------------------------------


FEATURE_DIR = f"{PROJECT_DIR}/data/processed/features"
os.makedirs(FEATURE_DIR, exist_ok=True)

joblib.dump(X_features, f"{FEATURE_DIR}/X_features.pkl")
joblib.dump(y, f"{FEATURE_DIR}/y_labels.pkl")
joblib.dump(tfidf_vectorizer, f"{FEATURE_DIR}/tfidf_vectorizer.pkl")
joblib.dump(scaler, f"{FEATURE_DIR}/metadata_scaler.pkl")

metadata_df.to_csv(f"{FEATURE_DIR}/metadata_features.csv", index=False)
rule_features_df.to_csv(f"{FEATURE_DIR}/rule_features.csv", index=False)

print("Saved feature engineering outputs to:", FEATURE_DIR)

Saved feature engineering outputs to: /content/drive/MyDrive/Confidential_Risk_Project/data/processed/features


We started the feature engineering notebook by loading the final processed multi-label dataset from combined_dataset.csv. The dataset contains 66,910 text samples and 7 target risk labels which are financial risk, credential risk, customer information risk, proprietary risk, legal risk, attachment risk, and phishing/spam risk.
We then created three groups of input features for the machine learning models. First, we cleaned the raw text by lowercasing, removing noisy formatting artifacts, replacing URLs/emails/numbers with standard tokens, and reducing unnecessary symbols. Then, we created metadata features such as text length, word count, digit count, dollar-sign count, uppercase count, and punctuation counts. Third, we added rule-based indicator features that detect risk-related terms such as attachment extensions, financial language, credential terms, customer/account terms, legal terms, and internal/confidential routing phrases.
Finally, we generated TF-IDF text features using word and phrase patterns from the cleaned text. We improved the text cleaning after noticing noisy underscore/code-like tokens in the initial TF-IDF output. The final feature set combines TF-IDF features, metadata features, and rule-based indicators, and is ready to be saved for the model training notebook.